# Session 15 Solutions: Distributions, Outliers and the Titanic

Data Science with Python, Level 1, Stage 2: Working with Data

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
flats = pd.read_csv(DATA_URL + "apartments.csv")

titanic = sns.load_dataset("titanic")

## In-Class Exercises

### In-Class Exercise 1: Area Distribution

In [ ]:
sns.histplot(data=flats, x="area_m2", bins=30)
plt.title("Apartment sizes")
plt.show()
print(round(flats["area_m2"].mean(), 1), flats["area_m2"].median(), round(flats["area_m2"].skew(), 2))

### In-Class Exercise 2: Cap and Compare

In [ ]:
bikes = pd.read_csv(DATA_URL + "bike_rentals.csv")
lo, hi = bikes["rentals"].quantile([0.01, 0.99])
capped = bikes["rentals"].clip(lower=lo, upper=hi)
print(round(bikes["rentals"].mean(), 1), round(capped.mean(), 1))
print(round(bikes["rentals"].std(), 1), round(capped.std(), 1))

### In-Class Exercise 3: Children by Class

In [ ]:
children = titanic[titanic["who"] == "child"]
children.groupby("class")["survived"].agg(["count", "mean"]).round(2)

### In-Class Exercise 4: Family Size

In [ ]:
titanic["family_size"] = titanic["sibsp"] + titanic["parch"] + 1
sns.barplot(data=titanic, x="family_size", y="survived")
plt.title("Survival rate by family size")
plt.show()

## Part A: Basic

### Exercise 1: Mean or Median?

In [ ]:
for col in ["area_m2", "building_age", "distance_center_km", "price_k"]:
    s = flats[col]
    print(f"{col:<20} mean {s.mean():7.1f}  median {s.median():7.1f}  skew {s.skew():5.2f}")
# Report the median for price_k, which is strongly skewed; the others are close to symmetric.

### Exercise 2: Basic Rates

In [ ]:
print(titanic["survived"].value_counts())
print(round(titanic["survived"].mean(), 3))
older = titanic[titanic["age"] > 60]
print(len(older), round(older["survived"].mean(), 3))

## Part B: Intermediate

### Exercise 3: Exam Score Outliers

In [ ]:
exams = pd.read_csv(DATA_URL + "exam_scores.csv")
m = exams["math"]
q1, q3 = m.quantile([0.25, 0.75])
iqr = q3 - q1
iqr_mask = (m < q1 - 1.5 * iqr) | (m > q3 + 1.5 * iqr)
z_mask = ((m - m.mean()) / m.std()).abs() > 3
print(exams.loc[iqr_mask, ["math", "study_hours_week", "attendance_pct"]])
print(z_mask.sum(), "with |z| > 3")
# The very low and very high scores mostly match very little or very much study time.

### Exercise 4: A Pivot Table of Survival

In [ ]:
pd.pivot_table(titanic, index="class", columns="who", values="survived", aggfunc="mean", margins=True).round(2)
# Women and children survived far more often than men in every class.
# Even children in third class survived less often than children in first and second class.

## Part C: Challenge

### Exercise 5: Fill the Missing Ages

In [ ]:
filled = titanic["age"].fillna(titanic.groupby(["class", "who"])["age"].transform("median"))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(titanic["age"].dropna(), bins=30, ax=axes[0])
axes[0].set_title("Known ages")
sns.histplot(filled, bins=30, ax=axes[1])
axes[1].set_title("After filling by group")
plt.show()
# Group medians respect that children are young and first class passengers were older, so the filled values are more realistic.

## Part D: Cumulative Project: City Bikes

### Exercise 6: City Bikes: Unusual Days

In [ ]:
bikes = pd.read_csv(DATA_URL + "bike_rentals.csv")
g = bikes.groupby(["season", "weather"])["rentals"]
q1, q3 = g.transform(lambda s: s.quantile(0.25)), g.transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
bikes["unusual"] = (bikes["rentals"] < q1 - 1.5 * iqr) | (bikes["rentals"] > q3 + 1.5 * iqr)
print(bikes.loc[bikes["unusual"], ["date", "season", "weather", "temp_c", "wind_kmh", "rentals"]])
sns.scatterplot(data=bikes, x="temp_c", y="rentals", hue="unusual", s=15)
plt.title("Unusual days for their season and weather")
plt.show()
# Several unusual low days are very windy, which keeps riders away even in good weather.
# Some are from early 2023, before the bike scheme grew, so they look low compared with 2024.

### Exercise 7: Plan Your Project 2

In [ ]:
# 1. Dataset: bike_rentals.csv, because the bike team wants to plan for next year.
# 2. Questions: How much did rentals grow from 2023 to 2024? How strongly does temperature drive rentals?
#    How much do rain and wind reduce rentals? Do working days differ from weekends?
# 3. Columns: date and rentals; temp_c; weather and wind_kmh; working_day.
# 4. First check: missing values, and that every date appears exactly once.
bikes = pd.read_csv(DATA_URL + "bike_rentals.csv")
print(bikes.shape)
print(bikes.isna().sum())
print("Repeated dates:", bikes["date"].duplicated().sum())

## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | B | The few large values pull the mean up. |
| 2 | B | z counts standard deviations from the mean. |
| 3 | B | A big group can have many survivors and still a low rate. |
| 4 | B | The mean of a 0 and 1 column is the share of 1s. |
| 5 | B | Honest reports state how much data each result is based on. |